# Pilot Test: Jupyter + Public LLM Memory Dump Feasibility

이 노트북은 사용자가 지정한 memory dump를 읽고, 로컬 분석 결과를 공개 LLM API에 연결하는 최소 동작 데모입니다.

In [1]:
from pathlib import Path
import json
import os
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / 'configs' / 'jupyter_ai_openrouter.env')

DUMP_PATH = os.environ.get('DUMP_PATH', '/home/taejin/Jupyter/data/memory/memory.vmem').strip()
VMLINUX_PATH = os.environ.get('VMLINUX_PATH', '').strip()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DUMP_PATH =', DUMP_PATH)
print('VMLINUX_PATH =', VMLINUX_PATH or '(not provided)')
print('dump exists =', Path(DUMP_PATH).exists() if DUMP_PATH else False)
print('OPENAI_API_KEY configured =', bool(os.environ.get('OPENAI_API_KEY')))


PROJECT_ROOT = /home/taejin/Jupyter/jupyter-ramdump-analyzer
DUMP_PATH = /home/taejin/Jupyter/data/memory/memory.vmem
VMLINUX_PATH = (not provided)
dump exists = True
OPENAI_API_KEY configured = True


## Imports and runtime options

In [2]:
from analysis_pipeline import FeasibilityOptions, run_feasibility_analysis
from llm_assistant import LLMAssistant
from memory_kernel_analyzer import build_analysis_context, summarize_findings

MODEL = os.environ.get('OPENAI_MODEL', 'nvidia/nemotron-3-super-120b-a12b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')
ALLOW_RAW_CHUNK_EXCERPT = True
GENERATE_NEXT_STEPS = False

assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('MODEL =', MODEL)
print('API_BASE =', API_BASE)
assistant.is_configured()

MODEL = nvidia/nemotron-3-super-120b-a12b:free
API_BASE = https://openrouter.ai/api/v1


True

## Step 1. Build local analysis context

In [3]:
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 2. Inspect chunk samples sent to the LLM

In [4]:
for sample in context.raw_chunk_samples[:3]:
    print(sample)
    print('-' * 80)

{'offset': 0, 'size': 256, 'hex_preview': '53ff00f053ff00f0c3e200f053ff00f053ff00f054ff00f0888400f053ff00f0', 'ascii_preview': 'S...S.......S...S...T.......S...........V...V...V...V...W...........M...A.......9...Y...."J.....Y.......n...S...S.......'}
--------------------------------------------------------------------------------
{'offset': 536870912, 'size': 256, 'hex_preview': '280e3f001b0129001f0530001b0129001f0530001b0129001f0530001f053000', 'ascii_preview': '(.?...)...0...)...0...)...0...0...)...0...0...)...0...)...0...)...0...)...0...)...0...0...0...0...)...0...)...0...0...).'}
--------------------------------------------------------------------------------
{'offset': 1073741824, 'size': 256, 'hex_preview': 'fe86fe2124ed8f0525bda90b25ce981122bd0b1425fd8d1a252d872025ce8526', 'ascii_preview': '...!$...%...%..."...%...%-. %..&"(....)%../%|."..8%..>%..D%l.".x"..P%=.V%].\\%].b%..h%}.n%M.t%..z%...%.."...%...%.."...L.'}
-------------------------------------------------------------------

## Step 3. Run the feasibility pipeline

In [5]:
options = FeasibilityOptions(
    dump_path=DUMP_PATH,
    allow_raw_chunk_excerpt=ALLOW_RAW_CHUNK_EXCERPT,
    raw_chunk_limit=2,
    generate_next_steps=GENERATE_NEXT_STEPS,
)
report = run_feasibility_analysis(DUMP_PATH, assistant, options)
report.llm_enabled

True

## Step 4. Review local findings

In [6]:
print(json.dumps(report.findings, ensure_ascii=False, indent=2)[:4000])

{
  "file_info": {
    "path": "/home/taejin/Jupyter/data/memory/memory.vmem",
    "size_bytes": 4294967296,
    "size_mb": 4096.0,
    "size_gb": 4.0
  },
  "kernel_versions": [
    "Linux version 6.5.0-41-generic (buildd@lcy02-amd64-120)"
  ],
  "panic_pattern_count": 2,
  "panic_patterns": [
    "kernel_oops:BUG:",
    "crashes:crash"
  ],
  "top_call_traces": [],
  "interesting_strings_preview": [
    "alloc magic is broken at %p: %lx",
    "out of memory",
    "grub_calloc",
    "grub_malloc",
    "grub_realloc",
    "grub_zalloc",
    "PXE-E20: BIOS extended memory copy error.",
    "PXE-E20: BIOS extended memory copy error.  AH ==",
    "libdconfsettings.so",
    "/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so"
  ],
  "network_summary": {
    "internal_ip_count": 1,
    "external_ip_count": 17,
    "interesting_ports": [
      "20 (FTP-data)",
      "22 (SSH)",
      "23 (Telnet)",
      "25 (SMTP)",
      "53 (DNS)",
      "443 (HTTPS)"
    ]
  },
  "process_summary"

## Step 5. Review LLM output

API 키가 없으면 이 셀은 `None` 또는 오류 메시지를 보여줍니다.

In [7]:
print(report.llm_analysis)
print('\n' + '=' * 80 + '\n')
print(report.next_steps)

**핵심 이상 징후**

| 이상 징후 | 관찰된 증거 | 의미/영향 |
|----------|------------|-----------|
| **커널 오ops/BUG** | `kernel_oops:BUG:` 패턴 2회 발견 | 커널 내부에서 예외가 발생해 오ops가 기록됨 → 시스템 비정상 종료 가능성 |
| **크래시 로그** | `crashes:crash` 패턴 | 오ops 이후 실제 크래시 덤프가 생성되었음을 시사 |
| **메모리 할당 마법수 손상** | `alloc magic is broken at %p: %lx` | 커널 슬랩/페이지 할당자의 내부 무결성 검사 실패 → 메모리 손상 또는 잘못된 freed 사용 |
| **Out‑of‑Memory (OOM)** | `out of memory` 문자열 | 시스템이 사용 가능한 메모리를 모두 소모해 OOM killer가 작동했거나 할당 실패가 발생 |
| **GRUB 메모리 할당 오류** | `grub_calloc`, `grub_malloc`, `grub_realloc`, `grub_zalloc` | 부트로더(GRUB)가 실행 중에 메모리 할당에 실패 → 부팅 과정 초기에 메모리 문제가 있었음을 암시 |
| **PXE‑E20 BIOS 확장 메모리 복사 오류** | `PXE-E20: BIOS extended memory copy error.` 및 변형 | NIC/펌웨어가 메모리를 복사하는 과정에서 하드웨어 수준의 오류 → RAM 불량, BIOS 설정 오류, 또는 펌웨어 버그 가능성 |
| **의심스러운 외부 연결 다수** | 외부 IP 17개, 흥미로운 포트(20,22,23,25,53,443) 다수 | 시스템이 비정상적으로 많은 외부 엔드포인트와 통신 → mögliche 악성코드/명령·제어(C2) 활동 또는 서비스 오용 |
| **라이브러리 경로 노출** | `/usr/lib/x86_64-linux-gnu/gio/modules/libdconfsettings.so` | 특정 GIO 모듈이 메모리 덤프에 노

## Step 6. Script generation example

In [8]:
if assistant.is_configured():
    script = assistant.generate_analysis_script(
        'memory dump에서 panic signal과 call trace 후보를 추출하는 Python 스크립트',
        findings=report.findings,
    )
    print(script)
else:
    print('OPENAI_API_KEY is not configured')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Chat history
chat_history = widgets.Output(layout={'border': '1px solid gray', 'height': '300px', 'overflow_y': 'auto', 'padding': '5px'})

# Input box and button
msg_input = widgets.Text(placeholder='Type a question about the memory dump...', layout=widgets.Layout(width='80%'))
send_btn = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='20%'))

def on_send_clicked(b):
    user_msg = msg_input.value.strip()
    if not user_msg:
        return
    # Display user message
    with chat_history:
        print(f'\nYou: {user_msg}')
    # Clear input
    msg_input.value = ''
    # Get LLM response
    try:
        response = assistant.chat(user_msg)
    except Exception as e:
        response = f'Error: {e}'
    with chat_history:
        print(f'Bot: {response}')

send_btn.on_click(on_send_clicked)

# Also respond to Enter key in the text box
def on_submit(sender):
    on_send_clicked(None)
msg_input.on_submit(on_submit)

display(widgets.HBox([msg_input, send_btn]))
display(chat_history)

# Initial greeting
with chat_history:
    print('Bot: Hello! Ask me anything about the memory dump analysis.')
